In [2]:
import pandas as pd

In [3]:
manual_annotations = pd.read_csv("data/fda_manual_valid_llm_indications/llm_indications_sample_50_manual.csv")

In [4]:
llm_annotations = pd.read_csv("data/fda_manual_valid_llm_indications/llm_indications_sample_50.csv")

In [5]:
llm_annotations.application_number.nunique()

50

In [6]:
eval_df = manual_annotations[['application_number','manual_disease_names']].merge(llm_annotations[['application_number','disease_from_indications']], on='application_number',how='left')

In [7]:
import pandas as pd
import re

# ---------- Helpers ----------
def normalize_split(x):
    if pd.isna(x):
        return []
    return [i.strip().lower() for i in str(x).split("|") if i.strip()]

def normalize_text(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


# ---------- Matching functions ----------
def exact_match_fn(a, b):
    return a == b

def fuzzy_match_fn(a, b):
    a_norm = normalize_text(a)
    b_norm = normalize_text(b)
    return a_norm in b_norm or b_norm in a_norm


# ---------- Metric computation ----------
def compute_metrics_row(manual_list, llm_list, match_type="exact", raw_manual=None, raw_llm=None):
    
    # ---- Handle NaN cases ----
    if pd.isna(raw_manual) and pd.isna(raw_llm):
        return 1.0, 1.0, 1.0   # perfect agreement
    
    if pd.isna(raw_manual) or pd.isna(raw_llm):
        return 0.0, 0.0, 0.0   # one missing → mismatch

    # ---- Normal matching ----
    if match_type == "exact":
        match_fn = exact_match_fn
    elif match_type == "fuzzy":
        match_fn = fuzzy_match_fn
    else:
        raise ValueError("match_type must be 'exact' or 'fuzzy'")

    matched_manual = set()
    matched_llm = set()

    for m in manual_list:
        for l in llm_list:
            if match_fn(m, l):
                matched_manual.add(m)
                matched_llm.add(l)

    precision = len(matched_llm) / len(llm_list) if llm_list else 0
    recall = len(matched_manual) / len(manual_list) if manual_list else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


# ---------- Apply ----------
def evaluate(df, match_type="exact"):
    df = df.copy()

    df["manual_list"] = df["manual_disease_names"].apply(normalize_split)
    df["llm_list"] = df["disease_from_indications"].apply(normalize_split)

    # ---------- Total counts ----------
    total_manual_entities = df["manual_list"].apply(len).sum()
    total_llm_entities = df["llm_list"].apply(len).sum()
    
    # ---------- Unique counts ----------
    unique_manual = set(item for sublist in df["manual_list"] for item in sublist)
    unique_llm = set(item for sublist in df["llm_list"] for item in sublist)
    
    # ---------- Print ----------
    print("\n=== ENTITY STATS ===")
    print("Total manual entities:", total_manual_entities)
    print("Total LLM entities:", total_llm_entities)
    
    print("\nUnique manual entities:", len(unique_manual))
    print("Unique LLM entities:", len(unique_llm))
    
    df[["precision", "recall", "f1"]] = df.apply(
        lambda row: pd.Series(
            compute_metrics_row(
                row["manual_list"],
                row["llm_list"],
                match_type=match_type,
                raw_manual=row["manual_disease_names"],
                raw_llm=row["disease_from_indications"]
            )
        ),
        axis=1
    )

    print(f"\n=== {match_type.upper()} MATCH ===")
    print("Precision:", df["precision"].mean())
    print("Recall:", df["recall"].mean())
    print("F1:", df["f1"].mean())

    return df

In [8]:
df_exact = evaluate(eval_df, match_type="exact")


=== ENTITY STATS ===
Total manual entities: 201
Total LLM entities: 186

Unique manual entities: 190
Unique LLM entities: 174

=== EXACT MATCH ===
Precision: 0.7155765027322404
Recall: 0.6595083542188804
F1: 0.6686519532544201


In [9]:
df_fuzzy = evaluate(eval_df, match_type="fuzzy")


=== ENTITY STATS ===
Total manual entities: 201
Total LLM entities: 186

Unique manual entities: 190
Unique LLM entities: 174

=== FUZZY MATCH ===
Precision: 0.8314663023679416
Recall: 0.7871988304093567
F1: 0.7885849901732255


In [109]:
df_fuzzy.head()

,application_number,manual_disease_names,disease_from_indications,manual_list,llm_list,precision,recall,f1
0,ANDA063305,cutaneous candidiasis,cutaneous candidiasis,[cutaneous candidiasis],[cutaneous candidiasis],1.0,1.0,1.000000
1,ANDA084349,tonic-clonic seizures | psychomotor seizures ...,tonic-clonic (grand mal) seizures|psychomotor ...,"[tonic-clonic seizures, psychomotor seizures,...","[tonic-clonic (grand mal) seizures, psychomoto...",1.0,1.0,1.000000
2,NDA050641,drug-resistant bacteria | infections,infections,"[drug-resistant bacteria, infections]",[infections],1.0,0.5,0.666667
3,NDA219293,Philadelphia chromosome positive chronic myelo...,Philadelphia chromosome positive chronic myelo...,[philadelphia chromosome positive chronic myel...,[philadelphia chromosome positive chronic myel...,1.0,1.0,1.000000
4,NDA022560,postmenopausal osteoporosis,postmenopausal osteoporosis,[postmenopausal osteoporosis],[postmenopausal osteoporosis],1.0,1.0,1.000000
